In [13]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, broadcast, rand, lit, cast

In [2]:
spark = SparkSession.builder.appName("Phase3").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 04:36:47 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/24 04:36:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 04:36:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
data_crowd = [
    ("Peasant 1", "Crownlands", "Targaryen"),
    ("Soldier A", "Stormlands", "Baratheon"),
    ("Squire B", "The Reach", "Tyrell"),
    ("Peasant 2", "Crownlands", "Targaryen"),
    ("Hedge Knight", "Flea Bottom", "None")
] * 1000

df_crowd = spark.createDataFrame(data_crowd, ["person_name", "region", "allegiance"])

data_kingsguard = [
    ("Targaryen", "Ser Roland Crakehall"),
    ("Baratheon", "Ser Willem Wylde")
]
df_kingsguard = spark.createDataFrame(data_kingsguard, ["royal_house", "assigned_guard"])

# Partitioning: `repartition()` vs `coalesce()`

A **partition** is a physical chunk of data stored in the memory (RAM) of a single Executor core. If your cluster has **100 cores** and your DataFrame has **100 partitions**, PySpark can process them in parallel with maximum efficiency.

## `repartition(n)`

`repartition()` performs a **full network shuffle**. It redistributes data evenly across the cluster by hashing and moving records between executors, creating exactly `n` partitions.

### Characteristics

- Triggers a full shuffle operation
- Redistributes data evenly across partitions
- Can increase or decrease the number of partitions
- More expensive due to network I/O

### When to Use

Use `repartition()` when:

- Your data is **heavily skewed**
- You need to **increase the number of partitions**
- You want to utilize **idle CPU cores**
- Data needs to be redistributed evenly across the cluster

**Example:**  
If your cluster has **200 cores** but your DataFrame only has **10 partitions**, increasing the partition count with `repartition()` can significantly improve parallelism.

```python
df = df.repartition(200)
```

---

## `coalesce(n)`

`coalesce()` avoids a full shuffle whenever possible. Instead of redistributing data across the cluster, it **merges existing partitions**, typically by collapsing partitions that already reside on the same Executor node.

### Characteristics

- Minimizes or avoids network shuffling
- Primarily used to **reduce** the number of partitions
- More efficient than `repartition()` when shrinking partitions
- Preserves existing data locality

### When to Use

Use `coalesce()` when:

- You want to **decrease** the number of partitions
- You are **writing data to disk**
- You want fewer output files

**Common Use Case:**  
Before writing data to Parquet or Delta format, use `coalesce()` to avoid generating thousands of tiny files.

```python
df.coalesce(10).write.parquet("/output/path")
```

---

## Quick Comparison

| Feature | `repartition()` | `coalesce()` |
|----------|----------------|--------------|
| Shuffle Required | ✅ Yes | ❌ Usually No |
| Increase Partitions | ✅ Yes | ❌ No |
| Decrease Partitions | ✅ Yes | ✅ Yes |
| Performance Cost | Higher | Lower |
| Data Redistribution | Evenly across cluster | Merge existing partitions |
| Typical Use Case | Fix skew, improve parallelism | Reduce output files before writing |

## Rule of Thumb

> **Use `repartition()` when you need better distribution or more parallelism.**
>
> **Use `coalesce()` when reducing partitions for efficient file output without paying the cost of a full shuffle.**

In [7]:
print(f"Original Partitions: {df_crowd.rdd.getNumPartitions()}")
df_repartitioned = df_crowd.repartition(50)
df_coalesced =  df_repartitioned.coalesce(10)
print(f"Repartitioned Partitions: {df_repartitioned.rdd.getNumPartitions()}")
print(f"Coalesced Partitions: {df_coalesced.rdd.getNumPartitions()}")

Original Partitions: 20


Repartitioned Partitions: 50
Coalesced Partitions: 10


## 2. Shuffling and Broadcast Joins

A **shuffle** occurs when data must move between Executors to complete an operation, such as a standard `join()`.

### Why Shuffling is Expensive

By default, PySpark often uses a **Sort-Merge Join** strategy:

1. The left table is partitioned based on the join key.
2. The join keys are hashed and sorted.
3. The corresponding data is sent across the network to specific Executors.
4. The same process is repeated for the right table.
5. Matching partitions are then joined.

Since data is transferred across the cluster, network I/O becomes a major bottleneck.

> **Network shuffling is one of the primary causes of slow PySpark jobs.**

---

### Broadcast Join

When one of the tables is small enough to fit into the memory of a single node, a **Broadcast Join** is typically much faster.

Instead of shuffling both tables:

1. The Driver serializes the entire small table.
2. A complete copy of the small table is sent to every Executor.
3. Each Executor keeps the small table in memory.
4. The large table is streamed through locally.
5. Joins are performed using in-memory lookups.

### Benefits

- Eliminates shuffling of the large table
- Reduces network traffic
- Improves join performance significantly
- Ideal for joining a very large table with a small lookup table

### Example

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    on="id",
    how="inner"
)
```

---

### Sort-Merge Join vs Broadcast Join

| Feature | Sort-Merge Join | Broadcast Join |
|----------|----------------|----------------|
| Network Shuffle | High | Minimal |
| Large Table Movement | Yes | No |
| Small Table Replication | No | Yes |
| Memory Requirement | Lower | Small table must fit in memory |
| Performance | Slower for large datasets | Much faster when one table is small |

### Rule of Thumb

> Use a **Broadcast Join** whenever one side of the join is small enough to fit comfortably in Executor memory. This avoids expensive network shuffles and can dramatically reduce job execution time.

In [ ]:
# EXPENSIVE: Standard SortMergeJoin. 
# df_crowd is shuffled across the network.
df_heavy_join = df_crowd.join(
    df_kingsguard, 
    df_crowd.allegiance == df_kingsguard.royal_house, 
    "left"
)
display(df_heavy_join.show(truncate=False, n=5))

# HIGHLY OPTIMIZED: BroadcastHashJoin.
# df_kingsguard is copied to all Executors. df_crowd does not move!
df_light_join = df_crowd.join(
    broadcast(df_kingsguard),
    df_crowd.allegiance == df_kingsguard.royal_house,
    "left"
)
display(df_light_join.show(truncate=False, n=5))

+------------+-----------+----------+-----------+--------------------+
|person_name |region     |allegiance|royal_house|assigned_guard      |
+------------+-----------+----------+-----------+--------------------+
|Hedge Knight|Flea Bottom|None      |NULL       |NULL                |
|Soldier A   |Stormlands |Baratheon |Baratheon  |Ser Willem Wylde    |
|Peasant 1   |Crownlands |Targaryen |Targaryen  |Ser Roland Crakehall|
|Peasant 2   |Crownlands |Targaryen |Targaryen  |Ser Roland Crakehall|
|Peasant 1   |Crownlands |Targaryen |Targaryen  |Ser Roland Crakehall|
+------------+-----------+----------+-----------+--------------------+
only showing top 5 rows


None

+------------+-----------+----------+-----------+--------------------+
|person_name |region     |allegiance|royal_house|assigned_guard      |
+------------+-----------+----------+-----------+--------------------+
|Peasant 1   |Crownlands |Targaryen |Targaryen  |Ser Roland Crakehall|
|Soldier A   |Stormlands |Baratheon |Baratheon  |Ser Willem Wylde    |
|Squire B    |The Reach  |Tyrell    |NULL       |NULL                |
|Peasant 2   |Crownlands |Targaryen |Targaryen  |Ser Roland Crakehall|
|Hedge Knight|Flea Bottom|None      |NULL       |NULL                |
+------------+-----------+----------+-----------+--------------------+
only showing top 5 rows


None

## 3. Memory Management: `cache()` vs `persist()`

PySpark uses **lazy evaluation**, meaning transformations are not executed immediately. Instead, Spark builds a **Directed Acyclic Graph (DAG)** of operations and only executes it when an **action** is triggered.

### The Problem

Consider the following workflow:

```python
filtered_df = crowd.filter(...)
joined_df = filtered_df.join(kingsguard, ...)
```

When an action is executed:

```python
joined_df.count()
```

Spark computes the entire DAG:

```text
Read Source
    ↓
Filter
    ↓
Join
    ↓
Count
```

If you immediately execute another action:

```python
joined_df.show()
```

Spark will recompute the entire DAG again from the original source.

```text
Read Source
    ↓
Filter
    ↓
Join
    ↓
Show
```

This repeated computation can be expensive, especially when the transformations involve large datasets, joins, or aggregations.

---

### `cache()`

`cache()` stores the DataFrame so Spark can reuse the computed result instead of rebuilding the DAG for every action.

```python
joined_df.cache()
```

After the first action:

```python
joined_df.count()
```

the computed partitions are stored, allowing subsequent actions to reuse them:

```python
joined_df.show()
```

without re-executing the entire lineage.

#### Characteristics

- Simplest way to cache data
- Uses Spark's default storage level
- In modern PySpark, equivalent to:

```python
df.persist(StorageLevel.MEMORY_AND_DISK)
```

- Stores data in memory when possible
- Spills remaining partitions to disk if memory is insufficient

---

### `persist()`

`persist()` provides explicit control over how Spark stores the DataFrame.

```python
from pyspark import StorageLevel

df.persist(StorageLevel.DISK_ONLY)
```

Common storage levels include:

| Storage Level | Description |
|--------------|-------------|
| `MEMORY_ONLY` | Store partitions only in RAM |
| `MEMORY_AND_DISK` | Store in RAM; overflow spills to disk |
| `DISK_ONLY` | Store only on disk |
| `MEMORY_ONLY_SER` | Store serialized data in memory to reduce space usage |
| `MEMORY_AND_DISK_SER` | Serialized storage with disk fallback |

### When to Use

Use `persist()` when:

- You need precise control over storage behavior
- Memory is limited
- Recomputing the DataFrame is expensive
- You want serialized storage to reduce memory consumption

---

### `cache()` vs `persist()`

| Feature | `cache()` | `persist()` |
|----------|----------|------------|
| Ease of Use | Simple | More flexible |
| Storage Level Selection | No | Yes |
| Default Behavior | `MEMORY_AND_DISK` | User-defined |
| Best For | General-purpose caching | Fine-grained optimization |

---

### Memory Considerations

Caching is not free.

When a DataFrame is cached:

```text
Executor Memory
┌───────────────────┐
│ Cached DataFrame  │
│ Cached DataFrame  │
│ Broadcast Tables  │
│ Task Execution    │
└───────────────────┘
```

Executor memory is shared among:

- Cached DataFrames
- Broadcast variables
- Shuffle data
- Task execution memory

If too many DataFrames are cached:

- Memory pressure increases
- Garbage collection becomes frequent
- Executors may spill excessively to disk
- Jobs may fail with **Out Of Memory (OOM)** errors

---

### Releasing Cached Data

Once a cached DataFrame is no longer needed, remove it from memory:

```python
df.unpersist()
```

This frees executor resources for other stages of the application.

---

### Rule of Thumb

> Cache or persist a DataFrame only when it will be reused multiple times and the cost of recomputation is significant.
>
> Always call `unpersist()` when the cached data is no longer needed to avoid unnecessary memory pressure on the cluster.

In [12]:
df_light_join.cache()# We cache the result of the optimized join so we don't recompute it
print(f"Total rows: {df_light_join.count()}")
df_light_join.filter(col("assigned_guard").isNotNull()).show(5)
df_light_join.unpersist() #free up the memory when not in use!


Total rows: 5000
+-----------+----------+----------+-----------+--------------------+
|person_name|    region|allegiance|royal_house|      assigned_guard|
+-----------+----------+----------+-----------+--------------------+
|  Peasant 1|Crownlands| Targaryen|  Targaryen|Ser Roland Crakehall|
|  Soldier A|Stormlands| Baratheon|  Baratheon|    Ser Willem Wylde|
|  Peasant 2|Crownlands| Targaryen|  Targaryen|Ser Roland Crakehall|
|  Peasant 1|Crownlands| Targaryen|  Targaryen|Ser Roland Crakehall|
|  Soldier A|Stormlands| Baratheon|  Baratheon|    Ser Willem Wylde|
+-----------+----------+----------+-----------+--------------------+
only showing top 5 rows


DataFrame[person_name: string, region: string, allegiance: string, royal_house: string, assigned_guard: string]

## 4. The Silent Killer: Data Skew and Salting

One of the most common causes of poor Spark performance is **data skew**.

### What Is Data Skew?

Data skew occurs when a small number of partitions receive a disproportionately large amount of data while other partitions receive very little.

A useful analogy is a Kubernetes cluster:

> Imagine 99% of incoming traffic being routed to a single Pod while the remaining Pods sit almost idle. The overloaded Pod becomes the bottleneck for the entire application.

The same thing can happen in Spark.

---

### How Data Skew Happens

Suppose we have a dataset containing people's allegiances:

```text
Targaryen  → 10,000 rows
Baratheon  → 10 rows
Stark      → 15 rows
Tyrell     → 12 rows
```

When Spark performs a shuffle operation such as:

```python
df.groupBy("allegiance").count()
```

it hashes the grouping key and sends all records for a particular key to the same partition.

```text
Partition 1 → Stark
Partition 2 → Baratheon
Partition 3 → Tyrell
Partition 4 → Targaryen (10,000 rows)
```

As a result:

- One Executor receives most of the data.
- Other Executors finish quickly and sit idle.
- The overloaded Executor becomes the bottleneck.
- Memory pressure increases.
- The stage may fail with an Out Of Memory (OOM) error.

### Why It's Dangerous

Spark jobs complete only when **all tasks** finish.

Even if 99 tasks finish in seconds, the entire stage must wait for the one overloaded task processing the skewed partition.

```text
Task 1   ✓
Task 2   ✓
Task 3   ✓
...
Task 99  ✓

Task 100 ⏳ (processing skewed data)

Job Status: Waiting...
```

This is why a single skewed key can dramatically increase execution time.

---

### The Solution: Salting

**Salting** distributes a heavily skewed key across multiple partitions by artificially creating multiple versions of that key.

Instead of:

```text
Targaryen
```

Spark sees:

```text
Targaryen_1
Targaryen_2
Targaryen_3
...
Targaryen_10
```

A random number (the **salt**) is appended to each occurrence of the skewed key.

### Before Salting

```text
Targaryen → Partition 4 (10,000 rows)
```

### After Salting

```text
Targaryen_1 → Partition 1
Targaryen_2 → Partition 2
Targaryen_3 → Partition 3
...
Targaryen_10 → Partition 10
```

The workload is now spread across multiple Executors instead of being concentrated on one.

---

### Typical Salting Workflow

#### Step 1: Add a Salt Column

```python
from pyspark.sql.functions import floor, rand, concat_ws

salted_df = df.withColumn(
    "salt",
    floor(rand() * 10)
)
```

#### Step 2: Create a Salted Key

```python
salted_df = salted_df.withColumn(
    "salted_key",
    concat_ws("_", "allegiance", "salt")
)
```

Example output:

```text
Targaryen_0
Targaryen_1
Targaryen_7
Targaryen_3
```

#### Step 3: Perform the First Aggregation

```python
partial_result = (
    salted_df
    .groupBy("salted_key")
    .count()
)
```

This distributes the workload across multiple partitions.

#### Step 4: Remove the Salt and Aggregate Again

```python
final_result = (
    partial_result
    .groupBy("allegiance")
    .sum("count")
)
```

The second aggregation combines the partial results to produce the correct final totals.

---

### Visualizing the Improvement

#### Without Salting

```text
Targaryen (10,000 rows)
        │
        ▼
   Partition 4
        │
        ▼
   Executor 4 🔥
```

#### With Salting

```text
Targaryen_1 ──► Executor 1
Targaryen_2 ──► Executor 2
Targaryen_3 ──► Executor 3
Targaryen_4 ──► Executor 4
...
Targaryen_10 ─► Executor 10
```

The workload is distributed more evenly, improving parallelism and reducing the risk of memory bottlenecks.

---

### Rule of Thumb

> If a small number of keys account for a large percentage of your data, expect data skew.
>
> Use salting to split heavily skewed keys into multiple partitions, allowing Spark to distribute the workload across the cluster and avoid executor hotspots.

In [ ]:
from pyspark.sql.functions import concat
# 1. Add a random salt (0 to 9) to the heavily skewed key
df_salted = df_crowd.withColumn(
    "salted_allegiance",
    concat(col("allegiance"), lit("_"), (rand() * 10).cast("int"))
)

# 2. First Aggregation: Distributed across 10 Executors instead of 1
df_partial_counts = df_salted.groupBy("allegiance","salted_allegiance").count()# we took two parameters to avoid string parse issue for the next step where we grouped on the basis of "allegiance" only. This is a common technique to avoid skewed data issues in Spark.

# 3. Second Aggregation: Summing up the small partial counts (Extremely fast)
df_final_counts = df_partial_counts.groupBy("allegiance").sum("count")

# print(f"Partial counts: {df_partial_counts.count()}")
# print(f"Total rows: {df_final_counts.count()}")

display(df_final_counts.show(truncate=False, n=5))
display(df_partial_counts.show(truncate=False, n=10))

+----------+----------+
|allegiance|sum(count)|
+----------+----------+
|None      |1000      |
|Baratheon |1000      |
|Targaryen |2000      |
|Tyrell    |1000      |
+----------+----------+



None

+----------+-----------------+-----+
|allegiance|salted_allegiance|count|
+----------+-----------------+-----+
|Tyrell    |Tyrell_1         |100  |
|Baratheon |Baratheon_7      |93   |
|Tyrell    |Tyrell_4         |94   |
|Targaryen |Targaryen_0      |199  |
|Tyrell    |Tyrell_3         |110  |
|Targaryen |Targaryen_1      |229  |
|None      |None_8           |98   |
|Baratheon |Baratheon_9      |102  |
|Targaryen |Targaryen_4      |192  |
|Tyrell    |Tyrell_9         |106  |
+----------+-----------------+-----+
only showing top 10 rows


None

26/06/24 13:27:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1578825 ms exceeds timeout 120000 ms
26/06/24 13:27:27 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/24 13:27:29 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$